<div style="font-size: 24px; line-height: 1.6;">

# Misleading Variables: Some Columns Are Traps

## Cleanup is not housekeeping — it is deciding what evidence belongs

</div>

<div style="font-size: 24px; line-height: 1.6;">

**Tip for instructors:** if the code editor or output text is still small for your room, use browser zoom (Ctrl/Cmd + `+`) or bump *Settings → Theme → Increase Code Font Size* in JupyterLab. Markdown text is already enlarged inline.

</div>

<div style="font-size: 24px; line-height: 1.6;">

### Imports

</div>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

<div style="font-size: 24px; line-height: 1.6;">

## The story

A churn dataset contains useful predictors (plan, tenure), identifiers (`customer_id`), dates, and variables that *happen after* churn (`refund_after_churn`). The latter look powerfully predictive, but they leak the answer.

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 1. Load the churn dataset

</div>

In [ ]:
churn = pd.read_csv("../data/misleading_variables_churn.csv")
churn.head()

In [ ]:
churn.info()

<div style="font-size: 24px; line-height: 1.6;">

## 2. Column audit with `df.columns`

Ask whether each column is an **identifier**, **outcome**, **predictor**, **date**, **proxy**, or **leak**.

</div>

In [ ]:
list(churn.columns)

<div style="font-size: 24px; line-height: 1.6;">

## 3. Type audit with `df.dtypes`

Types reveal disguised dates, booleans, and numbers.

</div>

In [ ]:
churn.dtypes

<div style="font-size: 24px; line-height: 1.6;">

## 4. Convert with `df.astype()`

Use `astype()` when the intended type is clear.

</div>

In [ ]:
churn["churned"] = churn["churned"].astype("bool")
churn["churned"].dtype

<div style="font-size: 24px; line-height: 1.6;">

## 5. Convert dates with `pd.to_datetime()`

</div>

In [ ]:
churn["signup_date"] = pd.to_datetime(churn["signup_date"])
churn["signup_date"].dtype

<div style="font-size: 24px; line-height: 1.6;">

## 6. Find suspicious correlations

A variable can look powerful because it leaks future information.

</div>

In [ ]:
numeric = churn.select_dtypes(include="number")
numeric.corr(numeric_only=True).round(3)

<div style="font-size: 24px; line-height: 1.6;">

Pay special attention to anything that correlates strongly with `churned`. Ask: *could this value have been known at decision time, or is it a consequence of churn?*

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 7. Cross-check a suspect with the outcome

`refund_after_churn` smells like a leak — by name alone.

</div>

In [ ]:
pd.crosstab(churn["churned"], churn["refund_after_churn"])

<div style="font-size: 24px; line-height: 1.6;">

If refunds only happen *after* churn, this column can perfectly predict the outcome — but only because the outcome already happened.

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 8. Drop columns with `df.drop()`

Dropping columns is an **analytical decision** that should be explained, not a default cleanup step.

</div>

In [ ]:
safe = churn.drop(columns=["customer_id", "refund_after_churn", "last_login_days_ago"])
list(safe.columns)

<div style="font-size: 24px; line-height: 1.6;">

Why each drop:

- `customer_id` — identifier, no predictive content
- `refund_after_churn` — happens after the outcome (**leak**)
- `last_login_days_ago` — measured at extraction time, may also leak

</div>

<div style="font-size: 24px; line-height: 1.6;">

## 9. Drop rows vs. drop columns

Same verb, very different consequence.

</div>

In [ ]:
print("Drop rows with any NA:  ", churn.dropna().shape)
print("Drop one column:        ", churn.drop(columns=["customer_id"]).shape)

<div style="font-size: 24px; line-height: 1.6;">

## Mini-lab: leakage hunt

</div>

In [ ]:
print(churn.columns.tolist())
print(churn.dtypes)
safe = churn.drop(columns=["customer_id", "refund_after_churn"])
print("Kept columns:", list(safe.columns))

<div style="font-size: 24px; line-height: 1.6;">

## Discussion

- For each remaining column, when in the customer's lifetime is its value known? Before churn, at churn, or after?
- Which columns would you keep for an honest churn-prediction EDA, and which would you justify dropping in writing?

</div>

<div style="font-size: 24px; line-height: 1.6;">

## Takeaway

Functions introduced / reinforced: `columns`, `dtypes`, `astype`, `to_datetime`, `select_dtypes`, `corr`, `drop`.

**Concept learned: not every column deserves to survive EDA.**

</div>